In [5]:
from openai import OpenAI
from pathlib import Path
import polars as pl
import os
import time
from dotenv import load_dotenv




In [ ]:
load_dotenv()

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

INPUT_PATH = Path("../data/cleaned/dramas_clean.ndjson")
OUTPUT_PATH = Path("../data/cleaned/dramas_with_vectors.parquet")


EMBEDDING_MODEL = "text-embedding-3-large"
DIMENSIONS = 1536          
BATCH_SIZE = 200
COST_PER_1M_TOKENS = 0.13  # standard / 0.065 for batch API

In [8]:
df = pl.read_ndjson(INPUT_PATH)
print(f"Loaded {df.height} dramas")

Loaded 1437 dramas


In [9]:
# Title and year are intentionally excluded — they are facts for SQL filters
df = df.with_columns(
    pl.format(
        "Genres: {}\nTags: {}\nSynopsis: {}",
        pl.col("genres").list.join(", "),
        pl.col("tags").list.join(", "),
        pl.col("synopsis")
    ).alias("context")
)

In [10]:
# Sanity check context lengths (~4 chars per token)
stats = df.select(
    min_chars=pl.col("context").str.len_chars().min(),
    max_chars=pl.col("context").str.len_chars().max(),
    mean_chars=pl.col("context").str.len_chars().mean().round(0),
)
print(stats)

shape: (1, 3)
┌───────────┬───────────┬────────────┐
│ min_chars ┆ max_chars ┆ mean_chars │
│ ---       ┆ ---       ┆ ---        │
│ u32       ┆ u32       ┆ f64        │
╞═══════════╪═══════════╪════════════╡
│ 235       ┆ 2071      ┆ 836.0      │
└───────────┴───────────┴────────────┘


In [13]:
# Cost estimate
texts = df["context"].to_list()

estimated_tokens = sum(len(t) // 4 for t in texts)
estimated_cost_standard = estimated_tokens / 1_000_000 * 0.13
estimated_cost_batch = estimated_tokens / 1_000_000 * 0.065

print(f"Texts to embed:          {len(texts):,}")
print(f"Estimated tokens:        ~{estimated_tokens:,}")
print(f"Estimated cost standard: ~${estimated_cost_standard:.4f}")
print(f"Estimated cost batch:    ~${estimated_cost_batch:.4f}")

Texts to embed:          1,437
Estimated tokens:        ~299,709
Estimated cost standard: ~$0.0390
Estimated cost batch:    ~$0.0195


In [14]:
def get_embeddings_in_batches(texts: list[str], batch_size: int = BATCH_SIZE) -> list:
    all_embeddings = []
    total_batches = (len(texts) + batch_size - 1) // batch_size

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        batch_num = i // batch_size + 1
        print(f"Batch {batch_num}/{total_batches} ({len(batch)} texts)...", end=" ")

        for attempt in range(3):
            try:
                response = client.embeddings.create(
                    input=batch,
                    model=EMBEDDING_MODEL,
                    dimensions=DIMENSIONS
                )
                batch_embeddings = [data.embedding for data in response.data]
                all_embeddings.extend(batch_embeddings)
                break

            except Exception as e:
                print(f"\n  Attempt {attempt + 1} failed: {e}")
                if attempt < 2:
                    time.sleep(5)
                else:
                    raise RuntimeError(f"Batch {batch_num} failed after 3 attempts") from e

    return all_embeddings

In [15]:
print("Starting embedding generation...")
embeddings = get_embeddings_in_batches(texts)
print(f"\nGot {len(embeddings)} embeddings")
print(f"Dimensions: {len(embeddings[0])}")

Starting embedding generation...
Batch 1/8 (200 texts)... Batch 2/8 (200 texts)... Batch 3/8 (200 texts)... Batch 4/8 (200 texts)... Batch 5/8 (200 texts)... Batch 6/8 (200 texts)... Batch 7/8 (200 texts)... Batch 8/8 (37 texts)... 
Got 1437 embeddings
Dimensions: 1536


In [16]:
df_final = (
    df.with_columns(
        pl.Series(name="embedding", values=embeddings)
    )
    .select([
        "mdl_id",
        "mdl_url", 
        "title",
        "native_title",
        "synopsis",
        "country",
        "episodes",
        "content_rating",
        "network",
        "year",
        "genres",
        "tags",
        "mdl_score",
        "watchers",
        "embedding"
    ])
)

print(f"Final shape: {df_final.shape}")
df_final.head(3)

Final shape: (1437, 15)


mdl_id,mdl_url,title,native_title,synopsis,country,episodes,content_rating,network,year,genres,tags,mdl_score,watchers,embedding
i64,str,str,str,str,str,i64,str,str,i64,list[str],list[str],f64,i64,list[f64]
53505,"""https://mydramalist.com/53505-…","""The Untamed Special Edition""","""陈情令特別剪辑版""","""A 20-episode cut of The Untame…","""China""",20,"""15+ - Teens 15 or older""","""Tencent Video""",2020,"[""mystery"", ""wuxia"", ""fantasy""]","[""bromance"", ""smart male lead"", … ""cultivation""]",9.0,38663,"[0.00524, -0.000109, … 0.012495]"
9025,"""https://mydramalist.com/9025-n…","""Nirvana in Fire""","""琅琊榜""","""In sixth-century China, the Em…","""China""",54,"""13+ - Teens 13 or older""","""BTV""",2015,"[""military"", ""historical"", … ""political""]","[""power struggle"", ""smart male lead"", … ""sibling rivalry""]",9.0,59925,"[-0.009061, 0.022008, … -0.005627]"
62295,"""https://mydramalist.com/62295-…","""When I Fly Towards You""","""当我飞奔向你""","""In the autumn of 2012, Su Zai …","""China""",24,"""13+ - Teens 13 or older""","""Youku""",2023,"[""comedy"", ""romance"", ""youth""]","[""friendship"", ""love at first sight"", … ""introverted male lead""]",9.0,97911,"[-0.021278, 0.023434, … -0.012778]"


In [17]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_final.write_parquet(OUTPUT_PATH)
print(f"Saved to {OUTPUT_PATH}")

Saved to ..\data\cleaned\dramas_with_vectors.parquet
